In [6]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv, find_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Annotated, Literal
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from pydantic import BaseModel, Field
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

In [7]:
load_dotenv(find_dotenv())

True

In [8]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [9]:
class JokeState(TypedDict):

    topic : str
    joke : str
    explanation : str
    

In [10]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [11]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [12]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [13]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the bank?\n\nBecause it *kneaded* some *dough*!',
 'explanation': 'This is a classic pun-based joke that relies on wordplay and double meanings!\n\nHere\'s the breakdown:\n\n1.  **"Kneaded" vs. "Needed":**\n    *   **Kneaded (for pizza):** When you make pizza, you have to "knead" the dough – physically work it with your hands to develop the gluten. This is a very common action associated with pizza.\n    *   **Needed (for the bank):** The word "kneaded" sounds exactly like "needed," which means to require or want something.\n\n2.  **"Dough" (for pizza) vs. "Dough" (for money):**\n    *   **Dough (for pizza):** This is the base of the pizza, made from flour, water, yeast, etc.\n    *   **Dough (for money):** "Dough" is a very common slang term for money.\n\n**Putting it together:**\n\nThe joke plays on the idea that the pizza *literally* deals with "kneading dough" (the food). But the punchline uses the sound-alike words to mean the p

In [14]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the bank?\n\nBecause it *kneaded* some *dough*!', 'explanation': 'This is a classic pun-based joke that relies on wordplay and double meanings!\n\nHere\'s the breakdown:\n\n1.  **"Kneaded" vs. "Needed":**\n    *   **Kneaded (for pizza):** When you make pizza, you have to "knead" the dough – physically work it with your hands to develop the gluten. This is a very common action associated with pizza.\n    *   **Needed (for the bank):** The word "kneaded" sounds exactly like "needed," which means to require or want something.\n\n2.  **"Dough" (for pizza) vs. "Dough" (for money):**\n    *   **Dough (for pizza):** This is the base of the pizza, made from flour, water, yeast, etc.\n    *   **Dough (for money):** "Dough" is a very common slang term for money.\n\n**Putting it together:**\n\nThe joke plays on the idea that the pizza *literally* deals with "kneading dough" (the food). But the punchline uses the sound-alike 

In [15]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the bank?\n\nBecause it *kneaded* some *dough*!', 'explanation': 'This is a classic pun-based joke that relies on wordplay and double meanings!\n\nHere\'s the breakdown:\n\n1.  **"Kneaded" vs. "Needed":**\n    *   **Kneaded (for pizza):** When you make pizza, you have to "knead" the dough – physically work it with your hands to develop the gluten. This is a very common action associated with pizza.\n    *   **Needed (for the bank):** The word "kneaded" sounds exactly like "needed," which means to require or want something.\n\n2.  **"Dough" (for pizza) vs. "Dough" (for money):**\n    *   **Dough (for pizza):** This is the base of the pizza, made from flour, water, yeast, etc.\n    *   **Dough (for money):** "Dough" is a very common slang term for money.\n\n**Putting it together:**\n\nThe joke plays on the idea that the pizza *literally* deals with "kneading dough" (the food). But the punchline uses the sound-alike

In [16]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': "Why did the pasta break up with the sauce?\n\nBecause it couldn't *marinara* anymore!",
 'explanation': 'This joke is a classic pun! Here\'s the breakdown:\n\n1.  **The Setup:** "Why did the pasta break up with the sauce?" This immediately sets up a scenario of a romantic relationship ending.\n\n2.  **The Punchline:** "Because it couldn\'t *marinara* anymore!"\n\n3.  **The Pun:**\n    *   **Marinara** is a common type of tomato sauce, often served with pasta. This is its literal meaning in the culinary world.\n    *   The word "marinara" sounds almost exactly like "marry her" when spoken quickly.\n\n4.  **Putting it Together:**\n    The joke plays on the sound-alike. The pasta broke up with the sauce because it couldn\'t "marry her" (the sauce) anymore, implying a loss of commitment or desire to take the relationship to the next level. The word "marinara" cleverly replaces "marry her," making it a food-related pun.\n\nSo, the humor comes from the unexpected

In [17]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the bank?\n\nBecause it *kneaded* some *dough*!', 'explanation': 'This is a classic pun-based joke that relies on wordplay and double meanings!\n\nHere\'s the breakdown:\n\n1.  **"Kneaded" vs. "Needed":**\n    *   **Kneaded (for pizza):** When you make pizza, you have to "knead" the dough – physically work it with your hands to develop the gluten. This is a very common action associated with pizza.\n    *   **Needed (for the bank):** The word "kneaded" sounds exactly like "needed," which means to require or want something.\n\n2.  **"Dough" (for pizza) vs. "Dough" (for money):**\n    *   **Dough (for pizza):** This is the base of the pizza, made from flour, water, yeast, etc.\n    *   **Dough (for money):** "Dough" is a very common slang term for money.\n\n**Putting it together:**\n\nThe joke plays on the idea that the pizza *literally* deals with "kneading dough" (the food). But the punchline uses the sound-alike 

In [18]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the bank?\n\nBecause it *kneaded* some *dough*!', 'explanation': 'This is a classic pun-based joke that relies on wordplay and double meanings!\n\nHere\'s the breakdown:\n\n1.  **"Kneaded" vs. "Needed":**\n    *   **Kneaded (for pizza):** When you make pizza, you have to "knead" the dough – physically work it with your hands to develop the gluten. This is a very common action associated with pizza.\n    *   **Needed (for the bank):** The word "kneaded" sounds exactly like "needed," which means to require or want something.\n\n2.  **"Dough" (for pizza) vs. "Dough" (for money):**\n    *   **Dough (for pizza):** This is the base of the pizza, made from flour, water, yeast, etc.\n    *   **Dough (for money):** "Dough" is a very common slang term for money.\n\n**Putting it together:**\n\nThe joke plays on the idea that the pizza *literally* deals with "kneading dough" (the food). But the punchline uses the sound-alike

## Time Travel

In [22]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1146df-a04b-65ec-8001-5b60f396324e"}})

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the bank?\n\nBecause it *kneaded* some *dough*!'}, next=('generate_explanation',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1146df-a04b-65ec-8001-5b60f396324e'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-02-28T06:23:20.379953+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1146df-5037-6973-8000-c34e7596a80c'}}, tasks=(PregelTask(id='ff5b4dda-7f38-4541-c02b-e1c13f31ec21', name='generate_explanation', path=('__pregel_pull', 'generate_explanation'), error=None, interrupts=(), state=None, result={'explanation': 'This is a classic pun-based joke that relies on wordplay and double meanings!\n\nHere\'s the breakdown:\n\n1.  **"Kneaded" vs. "Needed":**\n    *   **Kneaded (for pizza):** When you make pizza, you have to "knead" the dough – physically work it with your hands to develop the gluten. This is a very common act

In [23]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1146df-a04b-65ec-8001-5b60f396324e"}})

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the bank?\n\nBecause it *kneaded* some *dough*!',
 'explanation': 'This joke is a classic pun, playing on two words that sound alike but have different meanings, and another word with two different meanings:\n\n1.  **"Kneaded" vs. "Needed":**\n    *   **Kneaded (pronounced "need-ed"):** This is a verb that describes the process of working dough with your hands, which is essential for making pizza crust.\n    *   **Needed (pronounced "need-ed"):** This means to require something, often money.\n\n2.  **"Dough":**\n    *   **Dough (pizza context):** This refers to the unbaked mixture of flour, water, and other ingredients that forms the base of the pizza.\n    *   **Dough (slang context):** This is a common informal term for money.\n\n**How the joke works:**\n\nThe humor comes from the clever wordplay. When the pizza says it "kneaded some dough," it sounds like it\'s talking about:\n\n*   **Literally (in a pizza sense):** The act of pre

In [24]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the bank?\n\nBecause it *kneaded* some *dough*!', 'explanation': 'This joke is a classic pun, playing on two words that sound alike but have different meanings, and another word with two different meanings:\n\n1.  **"Kneaded" vs. "Needed":**\n    *   **Kneaded (pronounced "need-ed"):** This is a verb that describes the process of working dough with your hands, which is essential for making pizza crust.\n    *   **Needed (pronounced "need-ed"):** This means to require something, often money.\n\n2.  **"Dough":**\n    *   **Dough (pizza context):** This refers to the unbaked mixture of flour, water, and other ingredients that forms the base of the pizza.\n    *   **Dough (slang context):** This is a common informal term for money.\n\n**How the joke works:**\n\nThe humor comes from the clever wordplay. When the pizza says it "kneaded some dough," it sounds like it\'s talking about:\n\n*   **Literally (in a pizza sens

In [25]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1146df-a04b-65ec-8001-5b60f396324e", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1146ef-af9d-64ea-8002-f28d4508eeb2'}}

In [29]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the pizza go to the bank?\n\nBecause it *kneaded* some *dough*!', 'explanation': 'This joke is a classic example of a **pun**, relying on clever wordplay with words that sound alike but have different meanings.\n\nHere\'s the breakdown:\n\n1.  **"Why did the pizza go to the bank?"**\n    *   This sets up a silly, anthropomorphic image (a pizza acting like a human and going to a bank). The absurdity makes you expect a humorous explanation.\n\n2.  **"Because it *kneaded* some *dough*!"**\n    *   **"Kneaded" vs. "Needed":** The word "kneaded" (pronounced "need-ed") refers to the process of working and mixing dough, which is a crucial step in making pizza. The pun is that it sounds exactly like "needed," meaning "required" or "wanted."\n    *   **"Dough" (food) vs. "Dough" (money):**\n        *   "Dough" is the primary ingredient of pizza (the mixture of flour, water, etc., before it\'s baked). So, a pizza literally *is* dough, an

In [30]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1146ef-af9d-64ea-8002-f28d4508eeb2"}})

{'topic': 'samosa',
 'joke': 'Why did the pizza go to the bank?\n\nBecause it *kneaded* some *dough*!',
 'explanation': 'This is a classic pun joke that plays on words that sound alike but have different meanings:\n\n1.  **"Kneaded" vs. "Needed":**\n    *   **Kneaded:** This is what you do to *dough* when you\'re making pizza (or bread). You press and stretch it to develop the gluten.\n    *   **Needed:** This means "required" or "wanted."\n    *   The joke uses "kneaded" to sound like "needed."\n\n2.  **"Dough" (food ingredient) vs. "Dough" (slang for money):**\n    *   **Dough:** This is the primary ingredient for pizza crust.\n    *   **Dough:** This is a common slang term for money. (e.g., "I need some dough to buy groceries.")\n\n**Putting it together:**\n\nThe joke implies the pizza "needed" (wanted) some "dough" (money) from the bank, but it cleverly uses the word "kneaded" (what you do to pizza dough) and the double meaning of "dough" (the food and the money) to create a funny,

In [31]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the pizza go to the bank?\n\nBecause it *kneaded* some *dough*!', 'explanation': 'This is a classic pun joke that plays on words that sound alike but have different meanings:\n\n1.  **"Kneaded" vs. "Needed":**\n    *   **Kneaded:** This is what you do to *dough* when you\'re making pizza (or bread). You press and stretch it to develop the gluten.\n    *   **Needed:** This means "required" or "wanted."\n    *   The joke uses "kneaded" to sound like "needed."\n\n2.  **"Dough" (food ingredient) vs. "Dough" (slang for money):**\n    *   **Dough:** This is the primary ingredient for pizza crust.\n    *   **Dough:** This is a common slang term for money. (e.g., "I need some dough to buy groceries.")\n\n**Putting it together:**\n\nThe joke implies the pizza "needed" (wanted) some "dough" (money) from the bank, but it cleverly uses the word "kneaded" (what you do to pizza dough) and the double meaning of "dough" (the food and the money

## Fault Tolerance

In [32]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [33]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [34]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("Step 3 executed")
    return {"done": True}

In [35]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print(" Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print(" Kernel manually interrupted (crash simulated).")

In [ ]:
# 6. Re-run to show fault-tolerant resume
print("\n Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n Final State:", final_state)

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))